# Exercise 4 — store_ohlcv and load_ohlcv

`store_ohlcv` persists a validated, normalized DataFrame to SQLite. `load_ohlcv` reads it back. The `db_path` parameter must be a real file path for round-trip persistence — each `sqlite3.connect(':memory:')` opens a different in-memory database.

In [ ]:
import pandas as pd, math, sqlite3, tempfile, os

OHLCV_COLS = ["Open", "High", "Low", "Close", "Volume"]

def _synthetic(ticker="TEST", period="1y", interval="1d", n=50):
    prices = [100.0 * (1 + 0.3 * math.sin(i * 2 * math.pi / n)) for i in range(n)]
    dates  = pd.date_range("2023-01-01", periods=n, freq="B")
    close  = pd.Series(prices, index=dates)
    return pd.DataFrame({
        "Open":   close.shift(1).fillna(close.iloc[0]),
        "High":   close * 1.01,
        "Low":    close * 0.99,
        "Close":  close,
        "Volume": pd.Series([1_000_000 + i * 1_000 for i in range(n)], index=dates),
    })
def validate_ohlcv(df):
    if not isinstance(df, pd.DataFrame): return False, "not a DataFrame"
    missing = [c for c in OHLCV_COLS if c not in df.columns]
    if missing: return False, "missing columns: " + ", ".join(missing)
    if len(df) == 0: return False, "DataFrame is empty"
    valid = df.dropna(subset=["High", "Low"])
    if len(valid) > 0 and (valid["High"] < valid["Low"]).any():
        return False, "High < Low detected"
    return True, ""
def normalize_ohlcv(df):
    df = df.copy()
    if not isinstance(df.index, pd.DatetimeIndex): df.index = pd.to_datetime(df.index)
    if df.index.tz is not None: df.index = df.index.tz_localize(None)
    return df[[c for c in OHLCV_COLS if c in df.columns]]
def fetch_ohlcv(ticker, period="1y", interval="1d", fetch_fn=None):
    if fetch_fn is not None: return fetch_fn(ticker, period, interval)
    import yfinance as yf
    df = yf.download(ticker, period=period, interval=interval, progress=False)
    if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)
    return df

# ── Exercise: implement store_ohlcv and load_ohlcv ───────────────────────────

def store_ohlcv(df, ticker, db_path=":memory:"):
    """Validate, normalize, and persist OHLCV data to SQLite.

    Table schema: ohlcv(ticker, date, open, high, low, close, volume)
    Primary key : (ticker, date) — INSERT OR REPLACE handles updates.

    Args:
        df      : pd.DataFrame with OHLCV columns
        ticker  : str, e.g. "AAPL"
        db_path : SQLite file path; use a real file for persistence.

    Returns:
        int — number of rows stored

    Raises:
        ValueError if validate_ohlcv(df) fails
    """
    # TODO:
    # 1. ok, reason = validate_ohlcv(df); if not ok: raise ValueError(...)
    # 2. df = normalize_ohlcv(df)
    # 3. con = sqlite3.connect(db_path)
    # 4. con.execute("CREATE TABLE IF NOT EXISTS ohlcv (...)")
    # 5. for date, row in df.iterrows(): build rows list
    # 6. con.executemany("INSERT OR REPLACE INTO ohlcv VALUES (?,?,?,?,?,?,?)", rows)
    # 7. con.commit(); con.close(); return len(rows)
    return 0


def load_ohlcv(ticker, db_path=":memory:"):
    """Load stored OHLCV data for ticker from SQLite.

    Returns:
        pd.DataFrame with DatetimeIndex and columns Open, High, Low, Close, Volume.
        Empty DataFrame (columns = OHLCV_COLS) if ticker not found.
    """
    # TODO:
    # 1. con = sqlite3.connect(db_path)
    # 2. rows = con.execute("SELECT date, open, high, low, close, volume
    #                        FROM ohlcv WHERE ticker=? ORDER BY date", (ticker,)).fetchall()
    # 3. catch sqlite3.OperationalError -> return empty DataFrame
    # 4. if not rows: return empty DataFrame
    # 5. build DataFrame, set index to pd.to_datetime("date" column), drop "date" col
    # 6. return df
    return pd.DataFrame(columns=OHLCV_COLS)


### Checks

In [ ]:
checks = 0

# 1 — store_ohlcv raises ValueError for invalid data
try:
    try:
        store_ohlcv(pd.DataFrame(), "TEST")
        print("❌ 1: should have raised ValueError")
    except ValueError as ve:
        checks += 1; print("✅ 1 store_ohlcv raises ValueError for invalid data")
except Exception as e:
    print("❌ 1:", e)

# 2 — store_ohlcv returns the correct row count
try:
    with tempfile.NamedTemporaryFile(suffix=".db", delete=False) as f:
        _db2 = f.name
    try:
        df = _synthetic()
        n = store_ohlcv(df, "TEST", _db2)
        assert n == 50, f"expected 50 rows, got {n}"
        checks += 1; print("✅ 2 store_ohlcv returns correct row count")
    finally:
        os.unlink(_db2)
except Exception as e:
    print("❌ 2:", e)

# 3 — round-trip: stored data loads back correctly
try:
    with tempfile.NamedTemporaryFile(suffix=".db", delete=False) as f:
        _db3 = f.name
    try:
        df = _synthetic()
        store_ohlcv(df, "AAPL", _db3)
        loaded = load_ohlcv("AAPL", _db3)
        assert len(loaded) == 50, f"expected 50 rows, got {len(loaded)}"
        assert isinstance(loaded.index, pd.DatetimeIndex)
        assert "Close" in loaded.columns
        assert abs(round(loaded["Close"].iloc[0], 4) - round(df["Close"].iloc[0], 4)) < 1e-4
        checks += 1; print("✅ 3 round-trip: store then load produces matching data")
    finally:
        os.unlink(_db3)
except Exception as e:
    print("❌ 3:", e)

# 4 — multiple tickers stored and loaded independently
try:
    with tempfile.NamedTemporaryFile(suffix=".db", delete=False) as f:
        _db4 = f.name
    try:
        df = _synthetic()
        store_ohlcv(df, "AAPL", _db4)
        store_ohlcv(df, "MSFT", _db4)
        aapl = load_ohlcv("AAPL", _db4)
        msft = load_ohlcv("MSFT", _db4)
        assert len(aapl) == 50 and len(msft) == 50
        checks += 1; print("✅ 4 multiple tickers stored and loaded independently")
    finally:
        os.unlink(_db4)
except Exception as e:
    print("❌ 4:", e)

# 5 — load for non-existent ticker returns empty DataFrame
try:
    with tempfile.NamedTemporaryFile(suffix=".db", delete=False) as f:
        _db5 = f.name
    try:
        empty = load_ohlcv("NOTEXIST", _db5)
        assert isinstance(empty, pd.DataFrame)
        assert len(empty) == 0
        assert set(empty.columns) == set(OHLCV_COLS)
        checks += 1; print("✅ 5 load for missing ticker returns empty DataFrame")
    finally:
        os.unlink(_db5)
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
